# Troubleshoot Power BI Scanner API — Datasource Details

**Purpose:** Diagnose exactly why the Scanner API (`/admin/workspaces/getInfo`)
is **not returning datasource details** (type, connectionDetails, gatewayId).

This notebook focuses **exclusively on the SCAN API**. It does NOT fall back to
the per-dataset Admin API — the goal is to make the Scanner itself work correctly.

## What this notebook checks

| Cell | Check |
|------|-------|
| 1 | Setup — acquire token, configure constants |
| 2 | Decode JWT — identity type, permissions, roles/scopes |
| 3 | Tenant settings — is "Enhance admin APIs" enabled? |
| 4 | Pick test workspaces — find workspaces that actually contain datasets |
| 5 | Scan WITH `datasourceDetails=True` — submit, poll, deep-inspect raw result |
| 6 | Scan WITHOUT `datasourceDetails=True` — compare to see if the flag matters |
| 7 | Deep structural walk — every dataset, every possible datasource key |
| 8 | Multi-workspace scan — test up to 5 workspaces to isolate per-workspace vs tenant-wide |
| 9 | Actionable diagnosis — root-cause checklist with fix instructions |

## Prerequisites

- Run in a **Microsoft Fabric notebook** (or Databricks with `mssparkutils`).
- Identity must have **Power BI Service Admin** or **Fabric Admin** role.
- A **default Lakehouse** is NOT required (this notebook only reads, never writes).

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# Cell 1 — Setup: acquire token, define constants and helpers
# ════════════════════════════════════════════════════════════════════════
print("[Cell 1] Starting setup ...")

import json, time, base64
import requests as http_requests

# ── Acquire token ────────────────────────────────────────────────────
try:
    pbi_token = mssparkutils.credentials.getToken(
        "https://analysis.windows.net/powerbi/api"
    )
except Exception:
    pbi_token = mssparkutils.credentials.getToken(
        "https://api.fabric.microsoft.com/"
    )

HEADERS = {
    "Authorization": f"Bearer {pbi_token}",
    "Content-Type": "application/json",
}
PBI_API = "https://api.powerbi.com/v1.0/myorg"

# ── Tuning constants ────────────────────────────────────────────────
REQUEST_TIMEOUT = 30        # seconds per HTTP call
POLL_INTERVAL   = 3         # seconds between status polls
MAX_POLL_TIME   = 120       # max seconds to wait for a single scan
MAX_TEST_WORKSPACES = 5     # Cell 8 — how many workspaces to compare

# ── Shared state (populated by later cells) ──────────────────────────
token_info          = {}
tenant_metadata_ok  = None          # True / False / None (could not check)
candidate_workspaces = []           # workspaces with datasets
scan_with_details   = None          # raw scan result WITH datasourceDetails
scan_without_details = None         # raw scan result WITHOUT datasourceDetails

print("[Cell 1] Token acquired. Setup complete.")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# Cell 2 — Decode JWT: identity, permissions, roles, scopes
# ════════════════════════════════════════════════════════════════════════
print("[Cell 2] Decoding JWT ...")

try:
    parts = pbi_token.split(".")
    if len(parts) < 2:
        raise ValueError("Token does not look like a JWT (less than 2 parts).")

    payload = parts[1] + "=" * (-len(parts[1]) % 4)
    decoded = json.loads(
        base64.urlsafe_b64decode(payload.encode("utf-8")).decode("utf-8")
    )

    upn   = decoded.get("upn")
    oid   = decoded.get("oid")
    appid = decoded.get("appid")
    roles = decoded.get("roles", [])
    scp   = decoded.get("scp", "")
    aud   = decoded.get("aud", "")

    if not isinstance(roles, list):
        roles = [str(roles)]

    identity_type = "service principal" if appid and not upn else "user"

    token_info = {
        "upn": upn,
        "oid": oid,
        "appid": appid,
        "roles": roles,
        "scp": scp,
        "aud": aud,
        "identity_type": identity_type,
    }

    print(f"  Identity type:  {identity_type}")
    print(f"  UPN / OID:      {upn or oid or '(not found)'}")
    print(f"  App ID:         {appid or '(not found)'}")
    print(f"  Audience:       {aud}")
    print(f"  Roles:          {roles}")
    print(f"  Scopes (scp):   {scp or '(none)'}")

    # ── Permission checks ────────────────────────────────────────────
    has_tenant_read = (
        "Tenant.Read.All" in roles
        or "Tenant.ReadWrite.All" in roles
        or "Tenant.Read.All" in str(scp)
        or "Tenant.ReadWrite.All" in str(scp)
    )
    print()
    print(f"  Tenant.Read.All present:  {'YES ✓' if has_tenant_read else 'NO ✗  ← Scanner API requires this'}")

    if not has_tenant_read:
        print()
        print("  ⚠  Without Tenant.Read.All the Scanner API will return 401/403.")
        print("     If you are a Fabric/PBI admin, the token may still work via")
        print("     delegated admin rights even without an explicit role claim.")

except Exception as e:
    print(f"  ERROR: {type(e).__name__}: {e}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# Cell 3 — Tenant settings: is "Enhance admin APIs" enabled?
# ════════════════════════════════════════════════════════════════════════
print("[Cell 3] Checking tenant settings ...")

try:
    url = f"{PBI_API}/admin/tenantSettings"
    resp = http_requests.get(url, headers=HEADERS, timeout=REQUEST_TIMEOUT)
    print(f"  HTTP {resp.status_code}")

    if resp.status_code in (401, 403):
        tenant_metadata_ok = None
        print("  Cannot read tenant settings (401/403). Skipping.")
        print("  This does NOT mean the setting is off — it only means")
        print("  this identity cannot read admin tenant settings.")
    elif resp.status_code != 200:
        tenant_metadata_ok = None
        print(f"  Unexpected: {resp.text[:300]}")
    else:
        data = resp.json()

        # The response may be {"tenantSettings": [...]} or a flat dict.
        settings_list = data.get("tenantSettings", data)
        if isinstance(settings_list, dict):
            settings_list = list(settings_list.values()) if settings_list else []

        enhance_setting = None
        scanner_keywords = []

        for item in (settings_list if isinstance(settings_list, list) else []):
            name = ""
            if isinstance(item, dict):
                name = (
                    item.get("settingName", "")
                    or item.get("title", "")
                    or str(item.get("name", ""))
                )
            elif isinstance(item, str):
                name = item
            else:
                continue

            name_lower = name.lower()

            # Exact match for the critical setting
            if "enhanceadminapisresponsewithdetailedmetadata" in name_lower.replace(" ", "").replace("_", ""):
                enhance_setting = item
            elif "metadata" in name_lower or "scanner" in name_lower or "enhanced" in name_lower:
                scanner_keywords.append(item)

        if enhance_setting is not None:
            if isinstance(enhance_setting, dict):
                enabled = enhance_setting.get("state", enhance_setting.get("enabled", "Unknown"))
                # "Enabled" / "Disabled" / True / False
                if str(enabled).lower() in ("enabled", "true"):
                    tenant_metadata_ok = True
                    print("  'Enhance admin APIs responses with detailed metadata': ENABLED ✓")
                else:
                    tenant_metadata_ok = False
                    print(f"  'Enhance admin APIs responses with detailed metadata': DISABLED ✗")
                    print()
                    print("  ⚠  THIS IS THE MOST COMMON ROOT CAUSE.")
                    print("     Without this setting, datasourceDetails=True has no effect.")
                    print("     The scan result will contain datasourceUsages (just IDs)")
                    print("     but the datasourceInstances array (with type, connectionDetails,")
                    print("     gatewayId) will be EMPTY or MISSING.")
                    print()
                    print("  FIX → Power BI Admin Portal > Tenant settings > Admin API settings")
                    print("        > 'Enhance admin APIs responses with detailed metadata' → Enable")
                print()
                print(f"  Raw setting object: {json.dumps(enhance_setting, indent=2)[:500]}")
            else:
                tenant_metadata_ok = None
                print(f"  Found setting but unexpected format: {enhance_setting}")
        else:
            tenant_metadata_ok = None
            print("  Could not find 'Enhance admin APIs responses with detailed metadata'")
            print("  in the tenant settings response.")
            if scanner_keywords:
                print("  Related settings found:")
                for s in scanner_keywords[:5]:
                    print(f"    {json.dumps(s, indent=2)[:300]}")

except Exception as e:
    tenant_metadata_ok = None
    print(f"  ERROR: {type(e).__name__}: {e}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# Cell 4 — Find workspaces that contain datasets (test candidates)
# ════════════════════════════════════════════════════════════════════════
print("[Cell 4] Fetching workspaces to find ones with datasets ...")

try:
    # Fetch up to 1000 workspaces (non-personal only)
    raw_groups = []
    top = 1000
    skip = 0
    while True:
        url = f"{PBI_API}/admin/groups?$top={top}&$skip={skip}"
        resp = http_requests.get(url, headers=HEADERS, timeout=REQUEST_TIMEOUT)
        if resp.status_code == 429:
            wait = int(resp.headers.get("Retry-After", 30))
            print(f"  Rate-limited. Sleeping {wait}s ...")
            time.sleep(wait)
            continue
        if resp.status_code != 200:
            print(f"  HTTP {resp.status_code}: {resp.text[:300]}")
            break
        batch = resp.json().get("value", [])
        if not batch:
            break
        raw_groups.extend(batch)
        if len(batch) < top:
            break
        skip += top

    print(f"  Total workspaces fetched: {len(raw_groups)}")

    # Exclude personal workspaces for cleaner diagnostics
    groups = [g for g in raw_groups if g.get("type", "") != "PersonalGroup"]
    print(f"  Non-personal workspaces: {len(groups)}")

    # ── Quick scan to find workspaces that have datasets ─────────────
    # We do a lightweight getInfo (NO datasourceDetails) to discover which
    # workspaces actually contain datasets, then pick the best candidates.
    # We scan in small batches to be fast.

    print("  Probing workspaces for datasets (quick scan) ...")
    candidate_workspaces.clear()
    probe_batch_size = 100
    probed = 0

    for batch_start in range(0, len(groups), probe_batch_size):
        if len(candidate_workspaces) >= MAX_TEST_WORKSPACES * 3:
            break  # we have enough candidates

        batch_ids = [g["id"] for g in groups[batch_start:batch_start + probe_batch_size]]
        probe_url = (
            f"{PBI_API}/admin/workspaces/getInfo"
            f"?datasourceDetails=False"
            f"&getArtifactUsers=False"
            f"&lineage=False"
            f"&datasetSchema=False"
            f"&datasetExpressions=False"
        )
        sub = http_requests.post(
            probe_url, headers=HEADERS, json={"workspaces": batch_ids},
            timeout=REQUEST_TIMEOUT,
        )
        if sub.status_code not in (200, 202):
            print(f"    Probe submit failed ({sub.status_code}): {sub.text[:200]}")
            continue

        sub_json = sub.json()
        scan_id = sub_json.get("id") or sub_json.get("scanId")
        if not scan_id:
            continue

        # Poll
        start_ts = time.time()
        status = ""
        while time.time() - start_ts < MAX_POLL_TIME:
            st = http_requests.get(
                f"{PBI_API}/admin/workspaces/scanStatus/{scan_id}",
                headers=HEADERS, timeout=REQUEST_TIMEOUT,
            )
            if st.status_code == 200:
                status = st.json().get("status", "").lower()
                if status in ("succeeded", "failed"):
                    break
            time.sleep(POLL_INTERVAL)

        if status != "succeeded":
            continue

        res = http_requests.get(
            f"{PBI_API}/admin/workspaces/scanResult/{scan_id}",
            headers=HEADERS, timeout=REQUEST_TIMEOUT,
        )
        if res.status_code != 200:
            continue

        for ws in res.json().get("workspaces", []):
            datasets = ws.get("datasets", [])
            if datasets:
                candidate_workspaces.append({
                    "id": ws.get("id", ""),
                    "name": ws.get("name", "(unnamed)"),
                    "dataset_count": len(datasets),
                })
        probed += len(batch_ids)

    # Sort: workspaces with MORE datasets first (more likely to have datasources)
    candidate_workspaces.sort(key=lambda w: w["dataset_count"], reverse=True)

    print(f"  Workspaces with datasets found: {len(candidate_workspaces)}")
    for i, ws in enumerate(candidate_workspaces[:MAX_TEST_WORKSPACES]):
        print(f"    [{i+1}] {ws['name']}  (datasets: {ws['dataset_count']}, id: {ws['id']})")

    if not candidate_workspaces:
        print()
        print("  ⚠  No workspaces with datasets found. Cannot continue diagnostics.")
        print("     Possible causes:")
        print("     • Token identity has no admin access to any workspace")
        print("     • Tenant has no published datasets")

except Exception as e:
    print(f"  ERROR: {type(e).__name__}: {e}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# Cell 5 — Scan WITH datasourceDetails=True  (primary diagnostic)
# ════════════════════════════════════════════════════════════════════════
print("[Cell 5] Scanning ONE workspace WITH datasourceDetails=True ...")

try:
    if not candidate_workspaces:
        raise ValueError("No candidate workspaces. Run Cell 4 first.")

    test_ws = candidate_workspaces[0]
    ws_id   = test_ws["id"]
    ws_name = test_ws["name"]
    print(f"  Target: {ws_name} ({ws_id})")

    # ── Submit scan ──────────────────────────────────────────────────
    scan_url = (
        f"{PBI_API}/admin/workspaces/getInfo"
        f"?datasourceDetails=True"
        f"&getArtifactUsers=True"
        f"&lineage=True"
        f"&datasetSchema=False"
        f"&datasetExpressions=False"
    )
    body = {"workspaces": [ws_id]}

    print(f"  POST URL: {scan_url}")
    print(f"  POST body: {json.dumps(body)}")

    sub_resp = http_requests.post(
        scan_url, headers=HEADERS, json=body, timeout=REQUEST_TIMEOUT,
    )
    print(f"  Submit HTTP {sub_resp.status_code}")
    print(f"  Submit response: {sub_resp.text[:500]}")

    if sub_resp.status_code not in (200, 202):
        raise RuntimeError(f"Scan submission failed (HTTP {sub_resp.status_code})")

    sub_json = sub_resp.json()
    scan_id  = sub_json.get("id") or sub_json.get("scanId")
    if not scan_id:
        raise RuntimeError(f"No scan ID in response: {sub_json}")

    print(f"  Scan ID: {scan_id}")

    # ── Poll ─────────────────────────────────────────────────────────
    start_ts = time.time()
    final_status = ""
    while True:
        elapsed = int(time.time() - start_ts)
        if elapsed > MAX_POLL_TIME:
            break
        st = http_requests.get(
            f"{PBI_API}/admin/workspaces/scanStatus/{scan_id}",
            headers=HEADERS, timeout=REQUEST_TIMEOUT,
        )
        st_json = st.json() if st.status_code == 200 else {}
        final_status = st_json.get("status", f"HTTP {st.status_code}")
        print(f"    Poll t={elapsed:03d}s → {final_status}")
        if str(final_status).lower() in ("succeeded", "failed"):
            break
        time.sleep(POLL_INTERVAL)

    if str(final_status).lower() != "succeeded":
        raise RuntimeError(f"Scan did not succeed. Last status: {final_status}")

    # ── Fetch result ─────────────────────────────────────────────────
    res = http_requests.get(
        f"{PBI_API}/admin/workspaces/scanResult/{scan_id}",
        headers=HEADERS, timeout=REQUEST_TIMEOUT,
    )
    print(f"  Result HTTP {res.status_code}")
    if res.status_code != 200:
        raise RuntimeError(f"Result fetch failed: {res.text[:500]}")

    full = res.json()
    workspaces = full.get("workspaces", [])
    if not workspaces:
        raise RuntimeError("scanResult contains no workspaces.")

    scan_with_details = workspaces[0]

    # ── Top-level inspection ─────────────────────────────────────────
    print()
    print("  ┌─── Scan Result (datasourceDetails=True) ───────────────")
    print(f"  │ Workspace keys: {sorted(scan_with_details.keys())}")

    ws_instances = scan_with_details.get("datasourceInstances", [])
    print(f"  │ workspace-level datasourceInstances: {len(ws_instances)}")

    datasets = scan_with_details.get("datasets", [])
    print(f"  │ datasets: {len(datasets)}")

    # ── Per-dataset breakdown ────────────────────────────────────────
    total_usages = 0
    total_ds_instances = 0
    total_ds_sources = 0
    datasets_with_usages = 0
    datasets_with_instances = 0

    for ds in datasets:
        usages    = ds.get("datasourceUsages", [])
        instances = ds.get("datasourceInstances", [])
        sources   = ds.get("datasources", [])
        total_usages += len(usages)
        total_ds_instances += len(instances)
        total_ds_sources += len(sources)
        if usages:
            datasets_with_usages += 1
        if instances or sources:
            datasets_with_instances += 1

    print(f"  │")
    print(f"  │ Across all {len(datasets)} datasets:")
    print(f"  │   datasourceUsages total:        {total_usages}  (in {datasets_with_usages} datasets)")
    print(f"  │   datasourceInstances (ds-level): {total_ds_instances}  (in {datasets_with_instances} datasets)")
    print(f"  │   datasources (ds-level):         {total_ds_sources}")
    print(f"  └────────────────────────────────────────────────────────")

    # ── Critical check: do usages have matching instances? ───────────
    print()
    if total_usages > 0 and len(ws_instances) == 0 and total_ds_instances == 0:
        print("  ⚠  PROBLEM DETECTED:")
        print("     datasourceUsages exist (dataset knows its datasources by ID)")
        print("     but datasourceInstances is EMPTY at both workspace and dataset level.")
        print("     This means the SCAN API returned references without details.")
        print()
        print("     ROOT CAUSE → 'Enhance admin APIs responses with detailed metadata'")
        print("     is almost certainly DISABLED in tenant settings.")
    elif total_usages > 0 and (len(ws_instances) > 0 or total_ds_instances > 0):
        print("  ✓  datasourceUsages AND datasourceInstances both present.")
        print("     The SCAN API is returning details correctly.")
    elif total_usages == 0 and len(datasets) > 0:
        print("  ⚠  No datasourceUsages found in any dataset.")
        print("     This workspace's datasets may not have external datasources,")
        print("     or they may be push/streaming datasets (which have no datasources).")

    # ── Print first workspace-level instance (if any) ────────────────
    if ws_instances:
        print()
        print("  First workspace-level datasourceInstance:")
        print(f"  {json.dumps(ws_instances[0], indent=4)}")

    # ── Print first dataset with usages ──────────────────────────────
    for ds in datasets:
        if ds.get("datasourceUsages"):
            print()
            print(f"  First dataset with usages: '{ds.get('name')}' (id: {ds.get('id')})")
            print(f"    Dataset keys: {sorted(ds.keys())}")
            print(f"    datasourceUsages ({len(ds['datasourceUsages'])}):")
            for u in ds["datasourceUsages"][:3]:
                print(f"      {json.dumps(u)}")
            ds_inst = ds.get("datasourceInstances", [])
            ds_src  = ds.get("datasources", [])
            if ds_inst:
                print(f"    datasourceInstances ({len(ds_inst)}):")
                for inst in ds_inst[:2]:
                    print(f"      {json.dumps(inst, indent=6)}")
            if ds_src:
                print(f"    datasources ({len(ds_src)}):")
                for src in ds_src[:2]:
                    print(f"      {json.dumps(src, indent=6)}")
            break

except Exception as e:
    print(f"  ERROR: {type(e).__name__}: {e}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# Cell 6 — Scan WITHOUT datasourceDetails=True  (comparison)
# ════════════════════════════════════════════════════════════════════════
print("[Cell 6] Scanning SAME workspace WITHOUT datasourceDetails ...")

try:
    if not candidate_workspaces:
        raise ValueError("No candidate workspaces. Run Cell 4 first.")

    test_ws = candidate_workspaces[0]
    ws_id   = test_ws["id"]
    ws_name = test_ws["name"]
    print(f"  Target: {ws_name} ({ws_id})")

    # ── Submit scan (datasourceDetails=False) ────────────────────────
    scan_url_no = (
        f"{PBI_API}/admin/workspaces/getInfo"
        f"?datasourceDetails=False"
        f"&getArtifactUsers=True"
        f"&lineage=True"
        f"&datasetSchema=False"
        f"&datasetExpressions=False"
    )
    body = {"workspaces": [ws_id]}

    sub_resp = http_requests.post(
        scan_url_no, headers=HEADERS, json=body, timeout=REQUEST_TIMEOUT,
    )
    if sub_resp.status_code not in (200, 202):
        raise RuntimeError(f"Submit failed (HTTP {sub_resp.status_code}): {sub_resp.text[:300]}")

    sub_json = sub_resp.json()
    scan_id  = sub_json.get("id") or sub_json.get("scanId")
    if not scan_id:
        raise RuntimeError(f"No scan ID in response: {sub_json}")

    # Poll
    start_ts = time.time()
    status = ""
    while time.time() - start_ts < MAX_POLL_TIME:
        st = http_requests.get(
            f"{PBI_API}/admin/workspaces/scanStatus/{scan_id}",
            headers=HEADERS, timeout=REQUEST_TIMEOUT,
        )
        if st.status_code == 200:
            status = st.json().get("status", "").lower()
            if status in ("succeeded", "failed"):
                break
        time.sleep(POLL_INTERVAL)

    if status != "succeeded":
        raise RuntimeError(f"Scan did not succeed. Last: {status}")

    res = http_requests.get(
        f"{PBI_API}/admin/workspaces/scanResult/{scan_id}",
        headers=HEADERS, timeout=REQUEST_TIMEOUT,
    )
    if res.status_code != 200:
        raise RuntimeError(f"Result fetch failed: {res.text[:500]}")

    scan_without_details = res.json().get("workspaces", [None])[0]
    if not scan_without_details:
        raise RuntimeError("No workspace in result.")

    # ── Compare ──────────────────────────────────────────────────────
    ws_inst_with    = len((scan_with_details or {}).get("datasourceInstances", []))
    ws_inst_without = len(scan_without_details.get("datasourceInstances", []))

    ds_with    = scan_with_details.get("datasets", []) if scan_with_details else []
    ds_without = scan_without_details.get("datasets", [])

    usages_with = sum(len(d.get("datasourceUsages", [])) for d in ds_with)
    usages_without = sum(len(d.get("datasourceUsages", [])) for d in ds_without)

    inst_with  = sum(len(d.get("datasourceInstances", [])) for d in ds_with)
    inst_without = sum(len(d.get("datasourceInstances", [])) for d in ds_without)

    print()
    print("  ┌─── Comparison: WITH vs WITHOUT datasourceDetails ──────")
    print(f"  │                                      WITH     WITHOUT")
    print(f"  │ WS-level datasourceInstances:     {ws_inst_with:>6}     {ws_inst_without:>6}")
    print(f"  │ Dataset-level datasourceUsages:    {usages_with:>6}     {usages_without:>6}")
    print(f"  │ Dataset-level datasourceInstances: {inst_with:>6}     {inst_without:>6}")
    print(f"  └────────────────────────────────────────────────────────")

    if ws_inst_with == 0 and ws_inst_without == 0 and inst_with == 0 and inst_without == 0:
        print()
        print("  ⚠  datasourceInstances is EMPTY in BOTH cases.")
        print("     The 'datasourceDetails=True' flag is having NO effect.")
        print("     This confirms the tenant setting is the problem.")
    elif ws_inst_with > ws_inst_without or inst_with > inst_without:
        print()
        print("  ✓  datasourceDetails=True produces MORE instance data.")
        print("     The flag IS working. The tenant setting appears enabled.")
    elif ws_inst_with == ws_inst_without and inst_with == inst_without:
        print()
        print("  ℹ  Both scans return the same instance counts.")
        print("     If counts are > 0, things work. If 0, the setting may be off")
        print("     or this workspace simply has no external datasources.")

except Exception as e:
    print(f"  ERROR: {type(e).__name__}: {e}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# Cell 7 — Deep structural walk: every dataset, every datasource key
# ════════════════════════════════════════════════════════════════════════
print("[Cell 7] Deep walk of scan_with_details ...")

try:
    if not scan_with_details:
        raise ValueError("scan_with_details is empty. Run Cell 5 first.")

    # ── Build a lookup from workspace-level datasourceInstances ──────
    ws_instance_map = {}
    for inst in scan_with_details.get("datasourceInstances", []):
        for key in ("datasourceId", "datasourceInstanceId", "id"):
            val = inst.get(key, "")
            if val:
                ws_instance_map[val] = inst

    print(f"  Workspace-level instance IDs indexed: {len(ws_instance_map)}")

    datasets = scan_with_details.get("datasets", [])
    print(f"  Datasets to inspect: {len(datasets)}")

    resolved_count   = 0
    unresolved_count = 0
    no_usages_count  = 0
    datasets_detail  = []

    for ds in datasets:
        ds_id   = ds.get("id", "")
        ds_name = ds.get("name", "(unnamed)")

        usages    = ds.get("datasourceUsages", [])
        instances = ds.get("datasourceInstances", [])
        sources   = ds.get("datasources", [])

        # Build dataset-level instance map
        ds_instance_map = {}
        for inst in instances:
            for key in ("datasourceId", "datasourceInstanceId", "id"):
                val = inst.get(key, "")
                if val:
                    ds_instance_map[val] = inst
        for src in sources:
            for key in ("datasourceId", "datasourceInstanceId", "id"):
                val = src.get(key, "")
                if val:
                    ds_instance_map[val] = src

        if not usages and not instances and not sources:
            no_usages_count += 1
            continue

        ds_resolved = 0
        ds_unresolved = 0
        unresolved_ids = []

        for usage in usages:
            uid = (
                usage.get("datasourceInstanceId")
                or usage.get("datasourceId")
                or usage.get("id", "")
            )
            if not uid:
                ds_unresolved += 1
                unresolved_ids.append("(empty ID)")
                continue

            # Check workspace-level, then dataset-level
            found = ws_instance_map.get(uid) or ds_instance_map.get(uid)
            if found and found.get("datasourceType"):
                ds_resolved += 1
            else:
                ds_unresolved += 1
                unresolved_ids.append(uid)

        # Also count direct instances/sources as resolved
        for src in instances + sources:
            if src.get("datasourceType"):
                ds_resolved += 1

        resolved_count += ds_resolved
        unresolved_count += ds_unresolved

        datasets_detail.append({
            "name": ds_name,
            "id": ds_id,
            "usages": len(usages),
            "ws_instances_matched": ds_resolved,
            "unresolved": ds_unresolved,
            "unresolved_ids": unresolved_ids[:5],
            "inline_instances": len(instances),
            "inline_sources": len(sources),
        })

    # ── Print summary table ──────────────────────────────────────────
    print()
    print(f"  Datasets with no datasource info at all: {no_usages_count}")
    print(f"  Datasets with some datasource info:      {len(datasets_detail)}")
    print(f"  Total resolved (type + connectionDetails): {resolved_count}")
    print(f"  Total UNRESOLVED (usage ID but no detail): {unresolved_count}")
    print()

    if datasets_detail:
        print("  Per-dataset breakdown (first 20):")
        print(f"  {'Dataset Name':<40} {'Usages':>7} {'Resolved':>9} {'Unresolved':>11} {'Inline':>7}")
        print(f"  {'─'*40} {'─'*7} {'─'*9} {'─'*11} {'─'*7}")
        for d in datasets_detail[:20]:
            print(
                f"  {d['name'][:40]:<40} {d['usages']:>7} "
                f"{d['ws_instances_matched']:>9} {d['unresolved']:>11} "
                f"{d['inline_instances'] + d['inline_sources']:>7}"
            )

    if unresolved_count > 0:
        print()
        print("  ⚠  Some datasourceUsages IDs could not be resolved to instances.")
        print("     Unresolved IDs (sample):")
        shown = set()
        for d in datasets_detail:
            for uid in d["unresolved_ids"]:
                if uid not in shown:
                    print(f"       • {uid}")
                    shown.add(uid)
                if len(shown) >= 10:
                    break
            if len(shown) >= 10:
                break

except Exception as e:
    print(f"  ERROR: {type(e).__name__}: {e}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# Cell 8 — Multi-workspace scan: is the problem workspace-specific?
# ════════════════════════════════════════════════════════════════════════
print("[Cell 8] Scanning multiple workspaces to compare ...")

try:
    test_list = candidate_workspaces[:MAX_TEST_WORKSPACES]
    if len(test_list) < 2:
        print("  Only 1 candidate workspace available — cannot compare.")
        print("  Skipping multi-workspace test.")
        raise SystemExit()

    ws_ids = [w["id"] for w in test_list]

    scan_url = (
        f"{PBI_API}/admin/workspaces/getInfo"
        f"?datasourceDetails=True"
        f"&getArtifactUsers=True"
        f"&lineage=True"
        f"&datasetSchema=False"
        f"&datasetExpressions=False"
    )
    body = {"workspaces": ws_ids}

    sub = http_requests.post(scan_url, headers=HEADERS, json=body, timeout=REQUEST_TIMEOUT)
    if sub.status_code not in (200, 202):
        raise RuntimeError(f"Submit failed ({sub.status_code}): {sub.text[:300]}")

    scan_id = sub.json().get("id") or sub.json().get("scanId")
    if not scan_id:
        raise RuntimeError("No scan ID returned")

    # Poll
    start_ts = time.time()
    status = ""
    while time.time() - start_ts < MAX_POLL_TIME:
        st = http_requests.get(
            f"{PBI_API}/admin/workspaces/scanStatus/{scan_id}",
            headers=HEADERS, timeout=REQUEST_TIMEOUT,
        )
        if st.status_code == 200:
            status = st.json().get("status", "").lower()
            if status in ("succeeded", "failed"):
                break
        time.sleep(POLL_INTERVAL)

    if status != "succeeded":
        raise RuntimeError(f"Scan did not succeed: {status}")

    res = http_requests.get(
        f"{PBI_API}/admin/workspaces/scanResult/{scan_id}",
        headers=HEADERS, timeout=REQUEST_TIMEOUT,
    )
    if res.status_code != 200:
        raise RuntimeError(f"Result fetch failed: {res.text[:500]}")

    multi_workspaces = res.json().get("workspaces", [])

    # ── Compare per workspace ────────────────────────────────────────
    print()
    print(f"  {'Workspace':<40} {'Datasets':>9} {'WS Inst':>8} {'Usages':>8} {'DS Inst':>8} {'DS Srcs':>8}")
    print(f"  {'─'*40} {'─'*9} {'─'*8} {'─'*8} {'─'*8} {'─'*8}")

    any_has_instances = False

    for ws in multi_workspaces:
        ws_name  = ws.get("name", "(unnamed)")[:40]
        datasets = ws.get("datasets", [])
        ws_inst  = len(ws.get("datasourceInstances", []))
        usages   = sum(len(d.get("datasourceUsages", [])) for d in datasets)
        ds_inst  = sum(len(d.get("datasourceInstances", [])) for d in datasets)
        ds_srcs  = sum(len(d.get("datasources", [])) for d in datasets)

        if ws_inst > 0 or ds_inst > 0 or ds_srcs > 0:
            any_has_instances = True

        print(f"  {ws_name:<40} {len(datasets):>9} {ws_inst:>8} {usages:>8} {ds_inst:>8} {ds_srcs:>8}")

    print()
    if any_has_instances:
        print("  ✓  At least one workspace has datasource instances.")
        print("     The SCAN API CAN return details. If some workspaces are missing")
        print("     them, those datasets may simply not have external datasources.")
    else:
        print("  ⚠  NO workspace has datasource instances.")
        print("     This is a TENANT-WIDE problem, not workspace-specific.")
        print("     The 'Enhance admin APIs' setting is very likely DISABLED.")

except SystemExit:
    pass
except Exception as e:
    print(f"  ERROR: {type(e).__name__}: {e}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# Cell 9 — Actionable diagnosis: root-cause checklist
# ════════════════════════════════════════════════════════════════════════
print("[Cell 9] Building diagnosis ...")
print()

checks = []

# ── 1. Token identity ────────────────────────────────────────────────
identity = token_info.get("identity_type", "(unknown)")
has_tenant_read = (
    "Tenant.Read.All" in token_info.get("roles", [])
    or "Tenant.ReadWrite.All" in token_info.get("roles", [])
    or "Tenant.Read.All" in str(token_info.get("scp", ""))
    or "Tenant.ReadWrite.All" in str(token_info.get("scp", ""))
)
checks.append({
    "name": "Token has Tenant.Read.All",
    "ok": has_tenant_read,
    "fix": (
        "Grant Tenant.Read.All (or Tenant.ReadWrite.All) to the app registration "
        "in Azure AD → API permissions → Power BI Service. "
        "If using a user identity, the user must be a Fabric/PBI Admin."
    ),
})

# ── 2. Tenant setting ───────────────────────────────────────────────
checks.append({
    "name": "Tenant setting 'Enhance admin APIs' is enabled",
    "ok": tenant_metadata_ok,
    "fix": (
        "Power BI Admin Portal → Tenant settings → Admin API settings → "
        "'Enhance admin APIs responses with detailed metadata' → ENABLE. "
        "This is the #1 reason datasourceInstances is empty in scan results."
    ),
})

# ── 3. Scan returns datasets ────────────────────────────────────────
has_datasets = bool(
    scan_with_details
    and scan_with_details.get("datasets")
)
checks.append({
    "name": "Scan result contains datasets",
    "ok": has_datasets,
    "fix": (
        "The scanned workspace has no datasets. Try a workspace that "
        "contains published datasets/semantic models."
    ),
})

# ── 4. Scan returns datasourceInstances ──────────────────────────────
has_instances = False
if scan_with_details:
    ws_inst = len(scan_with_details.get("datasourceInstances", []))
    ds_inst = sum(
        len(d.get("datasourceInstances", []))
        for d in scan_with_details.get("datasets", [])
    )
    has_instances = (ws_inst + ds_inst) > 0
checks.append({
    "name": "Scan returns datasourceInstances (with details)",
    "ok": has_instances,
    "fix": (
        "datasourceDetails=True was sent but no instances returned. "
        "Enable the 'Enhance admin APIs' tenant setting (see above). "
        "After enabling, wait a few minutes and re-run."
    ),
})

# ── 5. Usages can be resolved to instances ───────────────────────────
usages_exist = False
all_resolved = True
if scan_with_details:
    inst_map = {}
    for inst in scan_with_details.get("datasourceInstances", []):
        for k in ("datasourceId", "datasourceInstanceId", "id"):
            v = inst.get(k, "")
            if v:
                inst_map[v] = inst
    for ds in scan_with_details.get("datasets", []):
        for inst in ds.get("datasourceInstances", []) + ds.get("datasources", []):
            for k in ("datasourceId", "datasourceInstanceId", "id"):
                v = inst.get(k, "")
                if v:
                    inst_map[v] = inst
        for usage in ds.get("datasourceUsages", []):
            usages_exist = True
            uid = (
                usage.get("datasourceInstanceId")
                or usage.get("datasourceId")
                or usage.get("id", "")
            )
            if uid and uid not in inst_map:
                all_resolved = False

if usages_exist:
    checks.append({
        "name": "All datasourceUsage IDs resolve to instances",
        "ok": all_resolved,
        "fix": (
            "Some usage IDs have no matching datasourceInstance. "
            "This can happen if the tenant setting was recently enabled "
            "and the scan cache is stale. Wait 15-30 min and re-scan. "
            "Also verify the scan URL includes datasourceDetails=True."
        ),
    })

# ── Print checklist ──────────────────────────────────────────────────
print("=" * 70)
print("  SCANNER API DIAGNOSTIC CHECKLIST")
print("=" * 70)
print()

all_ok = True
for c in checks:
    status = c["ok"]
    if status is True:
        icon = "✓ PASS"
    elif status is False:
        icon = "✗ FAIL"
        all_ok = False
    else:
        icon = "? UNKNOWN"
        all_ok = False

    print(f"  [{icon}]  {c['name']}")
    if status is not True:
        print(f"          FIX: {c['fix']}")
    print()

print("=" * 70)
if all_ok:
    print("  All checks passed. The SCAN API should be returning full")
    print("  datasource details. If you are still missing data, verify")
    print("  that the specific datasets you care about actually have")
    print("  external datasources (push/streaming datasets do not).")
else:
    print("  One or more checks failed. Address the issues above and")
    print("  re-run this notebook.")
print("=" * 70)
print()
print(f"  Token identity:    {identity} ({token_info.get('upn') or token_info.get('oid') or '?'})")
print(f"  Tenant setting:    {tenant_metadata_ok}")
print(f"  Workspaces found:  {len(candidate_workspaces)}")
if scan_with_details:
    print(f"  Datasets in scan:  {len(scan_with_details.get('datasets', []))}")
    print(f"  WS instances:      {len(scan_with_details.get('datasourceInstances', []))}")